# Tahap 1: Pengumpulan Data (Data Ingestion) dengan Logika Finansial
Pada tahap awal ini, kita mengumpulkan dan menyiapkan *dataset* mentah. Karena ini adalah simulasi, kita men-generate 600.000 baris data *dummy* yang merepresentasikan profil finansial nasabah.

**Perbaikan Penting:** `status_kredit` ('Lancar' atau 'Macet') tidak lagi dibuat secara acak murni, melainkan dihitung berdasarkan probabilitas rasio beban utang terhadap pendapatan (Debt-to-Income/DTI). Semakin besar rasio utang seorang nasabah, semakin besar peluang kreditnya berstatus 'Macet'. Ini akan membuat data menjadi logis untuk dipelajari oleh model Machine Learning.

In [1]:
import pandas as pd
import numpy as np

print("Memuat 600.000 baris data simulasi logis...")
np.random.seed(42)
data_size = 600000 

# 1. Generate fitur dasar
pendapatan_tahunan = np.random.randint(30000000, 200000000, data_size)
total_utang_saat_ini = np.random.randint(0, 50000000, data_size)
jumlah_pinjaman_diajukan = np.random.randint(5000000, 100000000, data_size)
tenor_bulan = np.random.choice([12, 24, 36, 48, 60], data_size)
bunga_tahunan = np.random.uniform(0.05, 0.20, data_size)
status_pekerjaan = np.random.choice(['Pegawai Tetap', 'Kontrak', 'Freelance', 'Wirausaha'], data_size)

# 2. Membuat logika untuk target (status_kredit) agar realistis
# Estimasi kasar cicilan untuk menentukan DTI (Debt-to-Income) sementara
bunga_bulanan = bunga_tahunan / 12
estimasi_cicilan_baru = (jumlah_pinjaman_diajukan / tenor_bulan) + (jumlah_pinjaman_diajukan * bunga_bulanan)
estimasi_cicilan_lama = total_utang_saat_ini * 0.05
pendapatan_bulanan = pendapatan_tahunan / 12

# Kalkulasi rasio utang (DTI)
dti = (estimasi_cicilan_baru + estimasi_cicilan_lama) / pendapatan_bulanan

# Tentukan probabilitas macet berdasarkan beban utang
kondisi_berat = dti > 0.45       # Beban utang berat
kondisi_menengah = (dti > 0.30) & (dti <= 0.45) # Beban utang menengah
kondisi_ringan = dti <= 0.30     # Beban utang ringan

status_kredit = np.empty(data_size, dtype=object)

# Assign 'Macet' atau 'Lancar' berdasarkan probabilitas logis
# Jika beban berat, peluang macet 75%
status_kredit[kondisi_berat] = np.random.choice(['Lancar', 'Macet'], size=np.sum(kondisi_berat), p=[0.25, 0.75])
# Jika beban menengah, peluang macet 35%
status_kredit[kondisi_menengah] = np.random.choice(['Lancar', 'Macet'], size=np.sum(kondisi_menengah), p=[0.65, 0.35])
# Jika beban ringan, peluang macet hanya 5%
status_kredit[kondisi_ringan] = np.random.choice(['Lancar', 'Macet'], size=np.sum(kondisi_ringan), p=[0.95, 0.05])

# 3. Gabungkan menjadi DataFrame
df_raw = pd.DataFrame({
    'pendapatan_tahunan': pendapatan_tahunan,
    'total_utang_saat_ini': total_utang_saat_ini,
    'jumlah_pinjaman_diajukan': jumlah_pinjaman_diajukan,
    'tenor_bulan': tenor_bulan,
    'bunga_tahunan': bunga_tahunan,
    'status_pekerjaan': status_pekerjaan,
    'status_kredit': status_kredit
})

# Menyimpan dataset kotor 
nama_file_kotor = 'Dataset_Kotor_600k.xlsx' 
print(f"Menyimpan ke {nama_file_kotor} (tunggu beberapa menit)...")
df_raw.to_excel(nama_file_kotor, index=False)
print("Selesai! Silakan cek explorer VS Code untuk file excelnya.")

Memuat 600.000 baris data simulasi logis...
Menyimpan ke Dataset_Kotor_600k.xlsx (tunggu beberapa menit)...
Selesai! Silakan cek explorer VS Code untuk file excelnya.


# Tahap 2: Validasi Data Mentah (Exploratory Data Analysis / EDA)
Sebelum melakukan pembersihan, kita wajib mengetahui kondisi "kesehatan" data mentah kita. Langkah ini berfungsi sebagai radar untuk mendeteksi:
1. **Data Duplikat:** Baris data yang kembar dan berulang.
2. **Missing Values:** Sel data yang kosong.
3. **Anomali (Outlier):** Mengecek nilai minimum dan maksimum.
4. **Distribusi Kelas:** Memastikan proporsi kelas target (Lancar vs Macet). Laporan ini akan menjadi acuan untuk tahap *cleaning*.

In [2]:
print("=== LAPORAN VALIDASI DATA MENTAH ===")

# 1. Pengecekan Data Duplikat
jumlah_duplikat = df_raw.duplicated().sum()
print(f"\n1. Jumlah Baris Duplikat: {jumlah_duplikat} baris")
if jumlah_duplikat > 0:
    print("   -> Terdapat data kembar. Data ini harus dihapus di tahap Cleaning.")
else:
    print("   -> Aman! Tidak ada data kembar.")

# 2. Pengecekan Data Kosong (Missing Values)
jumlah_kosong = df_raw.isnull().sum().sum()
print(f"\n2. Jumlah Data Kosong (Missing Values): {jumlah_kosong} sel")

# 3. Pengecekan Anomali (Outlier) dengan Deskripsi Statistik
print("\n3. Ringkasan Statistik untuk Deteksi Anomali:")
ringkasan = df_raw[['pendapatan_tahunan', 'total_utang_saat_ini', 'jumlah_pinjaman_diajukan']].describe().loc[['min', 'mean', 'max']]
print(ringkasan)

# 4. Distribusi Kelas Target
print("\n4. Distribusi Status Kredit:")
print(df_raw['status_kredit'].value_counts(normalize=True) * 100)

=== LAPORAN VALIDASI DATA MENTAH ===

1. Jumlah Baris Duplikat: 0 baris
   -> Aman! Tidak ada data kembar.

2. Jumlah Data Kosong (Missing Values): 0 sel

3. Ringkasan Statistik untuk Deteksi Anomali:
      pendapatan_tahunan  total_utang_saat_ini  jumlah_pinjaman_diajukan
min         3.000058e+07          6.200000e+01              5.000076e+06
mean        1.149433e+08          2.498313e+07              5.250483e+07
max         1.999998e+08          4.999998e+07              9.999960e+07

4. Distribusi Status Kredit:
status_kredit
Lancar    60.386167
Macet     39.613833
Name: proportion, dtype: float64


# Tahap 3: Data Cleaning & Feature Engineering
Berdasarkan laporan EDA sebelumnya, kita mengeksekusi pembersihan data dengan membuang duplikat dan menambal data kosong (jika ada). 

Setelah data bersih, kita masuk ke tahap **Feature Engineering** untuk mengkalkulasi 3 variabel finansial baru yang krusial untuk mengukur risiko kredit:
* **Monthly Installment:** Estimasi cicilan bulanan (pokok + bunga).
* **Debt-to-Income Ratio (DTI):** Rasio beban utang terhadap pendapatan bulanan.
* **Remaining Income:** Sisa uang nasabah setelah dipotong kewajiban utang.

In [3]:
# Membuat salinan data
df_clean = df_raw.copy()

print("Melakukan Data Cleaning...")
# Menghapus duplikat dan mengisi nilai kosong
df_clean.drop_duplicates(inplace=True)
df_clean.fillna(df_clean.median(numeric_only=True), inplace=True)

print("Melakukan Feature Engineering...")
# A. Monthly Installment (Cicilan Bulanan)
bunga_bulanan = df_clean['bunga_tahunan'] / 12
df_clean['monthly_installment'] = (df_clean['jumlah_pinjaman_diajukan'] / df_clean['tenor_bulan']) + (df_clean['jumlah_pinjaman_diajukan'] * bunga_bulanan)

# B. Debt-to-Income Ratio (DTI)
pendapatan_bulanan = df_clean['pendapatan_tahunan'] / 12
estimasi_cicilan_lama = df_clean['total_utang_saat_ini'] * 0.05 
df_clean['debt_to_income_ratio'] = (df_clean['monthly_installment'] + estimasi_cicilan_lama) / pendapatan_bulanan

# C. Remaining Income (Sisa Pendapatan)
df_clean['remaining_income'] = pendapatan_bulanan - (df_clean['monthly_installment'] + estimasi_cicilan_lama)

print("Data Cleaning & Feature Engineering Selesai!")

Melakukan Data Cleaning...
Melakukan Feature Engineering...
Data Cleaning & Feature Engineering Selesai!


# Tahap 4: Encoding, Transformasi & Ekspor Objek (Joblib)
Tahap ini menyesuaikan format data agar bisa dibaca mesin dan adil secara skala:

1. **Label Encoding:** Mengubah kolom teks menjadi angka menggunakan variabel encoder yang dipisah untuk `status_pekerjaan` dan `status_kredit`.
2. **Standard Scaler:** Mengubah skala variabel numerik menjadi *Z-score* (menghilangkan satuan asli dan meratakan titik tengah menjadi 0).
3. **Ekspor Objek (Joblib Dump):** Menyimpan aturan encoding dan scaling ke dalam file `.joblib`. Ini wajib agar sistem aplikasi (Streamlit) mampu menerapkan aturan transformasi data yang sama persis pada input nasabah baru.

In [4]:
import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler

print("Melakukan Data Encoding dan Transformasi...")

# 1. Membuat variabel encoder terpisah
encoder_status_pekerjaan = LabelEncoder()
encoder_status_kredit = LabelEncoder()

# 2. Encoding teks ke angka
df_clean['status_pekerjaan_encoded'] = encoder_status_pekerjaan.fit_transform(df_clean['status_pekerjaan'])
df_clean['status_kredit_encoded'] = encoder_status_kredit.fit_transform(df_clean['status_kredit']) 

# Menghapus kolom teks yang sudah tidak terpakai
df_clean.drop(['status_pekerjaan', 'status_kredit'], axis=1, inplace=True)

# Normalisasi data numerik (Standardization)
scaler = StandardScaler()
kolom_numerik = ['pendapatan_tahunan', 'total_utang_saat_ini', 'jumlah_pinjaman_diajukan', 
                 'monthly_installment', 'debt_to_income_ratio', 'remaining_income']
df_clean[kolom_numerik] = scaler.fit_transform(df_clean[kolom_numerik])

# 3. MENYIMPAN OBJEK PREPROCESSING (Joblib Dump)
print("\n=== Menyimpan Objek Preprocessing ===")
joblib.dump(encoder_status_pekerjaan, 'encoder_status_pekerjaan.joblib')
joblib.dump(encoder_status_kredit, 'encoder_status_kredit.joblib')
joblib.dump(scaler, 'scaler_finansial.joblib')
print("3 File objek (.joblib) berhasil disimpan! Silakan cek folder explorer kamu.")

print("\n=== Proses Preprocessing Selesai! Preview Data: ===")
print(df_clean.head())

Melakukan Data Encoding dan Transformasi...

=== Menyimpan Objek Preprocessing ===
3 File objek (.joblib) berhasil disimpan! Silakan cek folder explorer kamu.

=== Proses Preprocessing Selesai! Preview Data: ===
   pendapatan_tahunan  total_utang_saat_ini  jumlah_pinjaman_diajukan  \
0           -0.574467             -0.540563                 -1.523670   
1            0.867735             -1.482126                  0.465834   
2            1.433833              0.214905                  0.095698   
3            1.275551             -1.131559                  0.169649   
4            0.983943              0.786168                 -1.001611   

   tenor_bulan  bunga_tahunan  monthly_installment  debt_to_income_ratio  \
0           12       0.170028            -0.763543             -0.557759   
1           60       0.097830            -0.469867             -0.844619   
2           12       0.198129             1.507809             -0.138506   
3           24       0.151637             0.2

# Tahap 5: Ekspor Dataset Bersih (Siap ML)
Data saat ini sudah 100% matang, bersih dari anomali, berbentuk numerik seutuhnya, dan diskalakan dengan baik. Langkah terakhir di *notebook* ini adalah mengekspor *dataframe* yang sudah rapi tersebut kembali ke dalam format `.xlsx`. File inilah yang nantinya akan di-*load* untuk melatih (*training*) model Machine Learning.

In [5]:
# 5. MENYIMPAN DATASET BERSIH
nama_file_bersih = 'Dataset_Bersih_Siap_ML_600k.xlsx' 

print(f"Menyimpan ke {nama_file_bersih} (tunggu beberapa menit)...")
df_clean.to_excel(nama_file_bersih, index=False)
print(f"Selesai! File {nama_file_bersih} berhasil dibuat.")
print("Data sudah matang dan siap dimasukkan ke notebook model Machine Learning!")

Menyimpan ke Dataset_Bersih_Siap_ML_600k.xlsx (tunggu beberapa menit)...
Selesai! File Dataset_Bersih_Siap_ML_600k.xlsx berhasil dibuat.
Data sudah matang dan siap dimasukkan ke notebook model Machine Learning!
